In [ ]:
### Part 3 — Machine Learning: predict no-shows and choose the model honestly ###
# -----------------------------------------------------------------------------------

# Task 3.1 — Features, split and baseline

import pandas as pd
from sklearn.model_selection import train_test_split

# load the CSV
df = pd.read_csv("appointments_clean.csv")

print("Dataset shape:", df.shape)

#Convert reminder_sent into numbers
df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

#Build the features (X)

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
]

#convert the categorical columns into numerical columns using one-hot encoding:

X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)


#Define the target (y)
y = df["no_show"]


#Split into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Required Outputs
print("Training size:", len(X_train))
print("Test size:", len(X_test))

print("No-shows in test set:", y_test.sum())

baseline_accuracy = (y_test == 0).mean()

print("Baseline accuracy:", round(baseline_accuracy, 4))
print("Baseline accuracy (%):", round(baseline_accuracy * 100, 2))


In [ ]:
##Task 3.2 — Logistic regression

# Import Libraries
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Scale the features
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

#Train Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

#Make predictions
y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]


#Evaluate the model
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPrecision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

# Print Model coefficients
coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": np.round(model.coef_[0], 3)
})

print(coefficients.to_string(index=False))


#Interpret the reminder coefficient
reminder_index = list(X.columns).index("reminder_sent")

reminder_coef = model.coef_[0][reminder_index]
reminder_sd = scaler.scale_[reminder_index]

odds_ratio_scaled = np.exp(reminder_coef)

print("\nReminder coefficient:", reminder_coef)
print("Reminder standard deviation:", reminder_sd)
print("Exp(coefficient):", odds_ratio_scaled)


#Calculate the real No-to-Yes effect
real_reminder_odds_ratio = np.exp(
    reminder_coef / reminder_sd
)

print(
    "\nOdds ratio for No to Yes:",
    round(real_reminder_odds_ratio, 3)
)

In [ ]:
# Task 3.3 — Decision tree and overfitting

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Full-depth Decision Tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
y_pred = dt_full.predict(X_test)

print("Full-depth Tree")
print("Training accuracy:", round(accuracy_score(y_train, dt_full.predict(X_train)), 3))
print("Test accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Tree depth:", dt_full.get_depth())


# Decision Tree with maximum depth 3
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)

print("\nDepth-3 Tree")
print("Training accuracy:", round(accuracy_score(y_train, dt_3.predict(X_train)), 3))
print("Test accuracy:", round(accuracy_score(y_test, dt_3.predict(X_test)), 3))

In [ ]:
#Task 3.4-Random forest and a comparison table


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

# Create model
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model
rf.fit(X_train, y_train)

# Predictions
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Evaluation
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Training accuracy:", round(rf.score(X_train, y_train), 3))
print("Test accuracy:", round(rf_accuracy, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))

# Find the top 3 Important Features
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance", ascending=False
)

print("\nTop 3 Important Features:\n")
print(importance.head(3).round(3))


In [ ]:
# Task 3.5 — Tune, choose the model and the threshold, then test once

#A)Tune the Random Forest using GridSearchCV

from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(rf_grid.best_score_, 3))

# B) Compare the four models using cross-validation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

#C) Calculate 5-fold cross-validation ROC-AUC
lr_cv = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

dt_cv = cross_val_score(
    DecisionTreeClassifier(max_depth=3, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv = rf_grid.best_score_

print("\nLogistic Regression:", round(lr_cv, 3))
print("Depth-3 Decision Tree:", round(dt_cv, 3))
print("Random Forest:", round(rf_cv, 3))
print("Tuned Random Forest:", round(tuned_rf_cv, 3))

#D)Choose the threshold using out-of-fold training predictions
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Out-of-fold probabilities for Logistic Regression
oof = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare two thresholds
for threshold in [0.5, 0.35]:

    oof_pred = (oof >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Precision:", round(precision_score(y_train, oof_pred), 3))
    print("Recall:", round(recall_score(y_train, oof_pred), 3))
    print("F1:", round(f1_score(y_train, oof_pred), 3))
    print("Number flagged:", int(oof_pred.sum()))


# E)Evaluate the final model on the test set


# Train the winning model
lr_pipe.fit(X_train, y_train)

# Get test probabilities
probs = lr_pipe.predict_proba(X_test)[:, 1]

# Use the selected threshold
threshold = 0.35
test_pred = (probs >= threshold).astype(int)

# Evaluate
print("\nTest Precision:", round(precision_score(y_test, test_pred), 3))
print("Test Recall:", round(recall_score(y_test, test_pred), 3))
print("Number flagged:", int(test_pred.sum()))
print("Test ROC-AUC:", round(roc_auc_score(y_test, probs), 3))

# Save test_risk_scores.csv

test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": probs,
    "flag": test_pred
})

test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print(test_risk_scores.head())
print("\nTotal rows:", len(test_risk_scores))
print("Total flagged:", test_risk_scores["flag"].sum())

In [ ]:
# Download the 'test_risk_scores.csv' file
from google.colab import files
#files.download("test_risk_scores.csv")